In [0]:
df_silver_facilities = spark.read.table("hdfc_ergo.hdfc_silver.dim_facilities")
df_gold_geography = spark.read.table("hdfc_ergo.hdfc_gold.dim_geography")

In [0]:
from pyspark.sql.functions import coalesce, lit, current_timestamp, row_number, col
from pyspark.sql.window import Window

# 1. Lookup geography_sk FROM Dim_Geography (Triple join for precision)
df_geo_lookup = df_gold_geography.select("geography_sk", "state", "city", "pincode").distinct()

df_gold = df_silver_facilities.join(
    df_geo_lookup,
    on=["state", "city", "pincode"],
    how="left"
)

# 2. Orphan Handling for Geography
df_gold = df_gold.withColumn("geography_sk", coalesce(col("geography_sk"), lit(-1)))

# 3. Generate Surrogate Key: AUTO_INCREMENT ordered by facility_id
window_spec = Window.orderBy("facility_id")
df_gold = df_gold.withColumn("facility_sk", row_number().over(window_spec))

# 4. Add SCD Type 2 Attributes
df_gold = df_gold.withColumn("_valid_from", current_timestamp()) \
                 .withColumn("_valid_to", lit(None).cast("timestamp")) \
                 .withColumn("_is_current", lit(True).cast("boolean"))

# 5. Select ONLY the columns required in Gold (Column Pruning)
# Drop raw PII/Address, drop replaced snowflake cols (state, city, pincode)
df_gold = df_gold.select(
    "facility_sk",
    "facility_id",                 # Business Key
    "hospital_license_number",     # Business Key
    "facility_name",
    "facility_type",
    "accreditation",               # Replaces nabh_accredited (more detailed)
    "teaching_hospital",
    "trauma_center_level",
    "emr_system",
    "network_status",
    "emergency_department",
    "icu_available",
    "beds_count",
    "surgical_suites",
    "cost_index",
    "nabh_rating",
    "patient_safety_grade",
    "readmission_rate",
    "mortality_rate",
    "active_status",
    "geography_sk",                # Replaces state, city, pincode
    "_valid_from",
    "_valid_to",
    "_is_current"
)

display(df_gold.limit(5))

In [0]:
gold_table_fqn = "hdfc_ergo.hdfc_gold.dim_facilities"

df_gold.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(gold_table_fqn)

print(f"✅ Successfully wrote Dim_Facilities to Gold layer.")